In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
df = spark.range(8)

df.show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
+---+



# Mini Project Analysis on Ecommerce Data

In [0]:
spark

# local[8] --- means 8 cores????

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
# Customers - 1mb -----> 1 pb
# orders - 1mb ----> 1pb

In [0]:
df = spark.read.format("csv").option("header", "true").option('inferSchema','true').load("dbfs:/FileStore/shared_uploads/mayank0953@gmail.com/customers.csv")
order_df = spark.read.format("csv").option("header", "true").load("dbfs:/FileStore/shared_uploads/mayank0953@gmail.com/orders.csv")

In [0]:
df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3|Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4|Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [0]:
# count Distinct City
from pyspark.sql.functions import *


In [0]:
df.select(countDistinct('city')).show()

+--------------------+
|count(DISTINCT city)|
+--------------------+
|                   8|
+--------------------+



In [0]:
active_customers = df.filter(col('is_active') == True) 

In [0]:
active_customers.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          1|Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     true|
|          2|Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     true|
|          7|Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|     true|
|          8|Customer_8|     Pune|  Karnataka|  India|       2023-06-22|     true|
|          9|Customer_9|   Mumbai|  Telangana|  India|       2023-01-05|     true|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [0]:
df.fillna({'city':'unknown'})

Out[18]: DataFrame[customer_id: int, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean]

In [0]:
# df.withColumn('registration_date',to_date(col('registration_date'),'yyyy-MM-dd'))

In [0]:
df = df.withColumn('registration_year',year(col('registration_date')))\
    .withColumn('registration_month',month(col('registration_date')))

In [0]:
df.display()

customer_id,name,city,state,country,registration_date,is_active,registration_year,registration_month
0,Customer_0,Pune,Maharashtra,India,2023-06-29,false,2023,6
1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,true,2023,12
2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,true,2023,10
3,Customer_3,Bangalore,Karnataka,India,2023-10-17,false,2023,10
4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,false,2023,3
5,Customer_5,Hyderabad,Karnataka,India,2023-07-28,false,2023,7
6,Customer_6,Pune,Delhi,India,2023-08-29,false,2023,8
7,Customer_7,Ahmedabad,West Bengal,India,2023-12-28,true,2023,12
8,Customer_8,Pune,Karnataka,India,2023-06-22,true,2023,6
9,Customer_9,Mumbai,Telangana,India,2023-01-05,true,2023,1


In [0]:
df.select(countDistinct('city')).collect()[0][0]

Out[31]: 8

In [0]:
# Count Customers by city

df.groupBy('city').count().orderBy(col('count').desc()).show(5)

+---------+-----+
|     city|count|
+---------+-----+
|     Pune| 2243|
|Hyderabad| 2242|
|  Kolkata| 2223|
|Bangalore| 2211|
|    Delhi| 2200|
+---------+-----+
only showing top 5 rows



In [0]:
# Count Customers by city

df.groupBy('state','country').count().orderBy(col('count').desc()).show(5)

+-----------+-------+-----+
|      state|country|count|
+-----------+-------+-----+
|      Delhi|  India| 2578|
|    Gujarat|  India| 2543|
| Tamil Nadu|  India| 2536|
|  Telangana|  India| 2520|
|West Bengal|  India| 2503|
+-----------+-------+-----+
only showing top 5 rows



In [0]:
# Pivot Table
df.groupBy('state').pivot('is_active').count().show()

+-----------+-----+----+
|      state|false|true|
+-----------+-----+----+
|  Karnataka| 1207|1276|
| Tamil Nadu| 1284|1252|
|    Gujarat| 1211|1332|
|      Delhi| 1356|1222|
|  Telangana| 1294|1226|
|Maharashtra| 1260|1230|
|West Bengal| 1306|1197|
+-----------+-----+----+



In [0]:
# Filter Customer of last 6 month
df.filter(col('state') == lit('Delhi')).display()

customer_id,name,city,state,country,registration_date,is_active,registration_year,registration_month
6,Customer_6,Pune,Delhi,India,2023-08-29,false,2023,8
18,Customer_18,Pune,Delhi,India,2023-10-04,true,2023,10
26,Customer_26,Delhi,Delhi,India,2023-03-22,true,2023,3
46,Customer_46,Kolkata,Delhi,India,2023-09-23,true,2023,9
54,Customer_54,Kolkata,Delhi,India,2023-12-13,false,2023,12
61,Customer_61,Hyderabad,Delhi,India,2023-12-31,false,2023,12
76,Customer_76,Kolkata,Delhi,India,2023-01-08,false,2023,1
82,Customer_82,Kolkata,Delhi,India,2023-09-07,false,2023,9
84,Customer_84,Pune,Delhi,India,2023-04-14,false,2023,4
92,Customer_92,Kolkata,Delhi,India,2023-06-25,true,2023,6


In [0]:
df.groupBy('state').agg(min('registration_date').alias('oldest_customer'),
                       max('registration_date').alias('newset')).show()

+-----------+---------------+----------+
|      state|oldest_customer|    newset|
+-----------+---------------+----------+
|  Karnataka|     2023-01-01|2023-12-31|
| Tamil Nadu|     2023-01-01|2023-12-31|
|    Gujarat|     2023-01-01|2023-12-31|
|      Delhi|     2023-01-01|2023-12-31|
|Maharashtra|     2023-01-01|2023-12-31|
|West Bengal|     2023-01-01|2023-12-31|
|  Telangana|     2023-01-01|2023-12-31|
+-----------+---------------+----------+



In [0]:
output_path = 'dbfs:/FileStore/shared_uploads/mayank0953@gmail.com/Processed'
df.write.mode('append').parquet(output_path)

In [0]:
df.count()

Out[53]: 17653

In [0]:
dbfs:/FileStore/shared_uploads/mayank0953@gmail.com/Processed/part-00000-tid-6898980444266488372-3361b90a-0b10-4ac5-911a-8fd085491a0b-70-1-c000.snappy.orc



In [0]:
'dbfs:/FileStore/shared_uploads/mayank0953@gmail.com/Processed/part-*'

# Orders
Single File Analysis on Orders

In [0]:
order_df.show(5)

+--------+-----------+----------+-----------------+---------+
|order_id|customer_id|order_date|     total_amount|   status|
+--------+-----------+----------+-----------------+---------+
|       0|       3692|2024-09-03|547.7160076008001|  Shipped|
|       1|      11055|2024-08-10|577.8942599188381|  Pending|
|       2|       6963|2024-08-22|484.2085562764487|  Pending|
|       3|      13268|2024-09-01|366.3286882431848|Cancelled|
|       4|       1131|2024-08-09|896.9588380686909|  Pending|
+--------+-----------+----------+-----------------+---------+
only showing top 5 rows



In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- registration_year: integer (nullable = true)
 |-- registration_month: integer (nullable = true)



In [0]:
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
order_df = order_df.withColumn('order_date',to_date('order_date','yyyy-MM-dd'))\
    .withColumn('total_amount',order_df.total_amount.cast('float'))

In [0]:
order_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- status: string (nullable = true)



In [0]:
customers_order_df = df.join(order_df,'customer_id','inner')

In [0]:
customers_order_df.display(5)

customer_id,name,city,state,country,registration_date,is_active,registration_year,registration_month,order_id,order_date,total_amount,status
2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,true,2023,10,10691,2024-06-16,215.94174,Pending
2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,true,2023,10,2859,2024-10-12,345.01572,Cancelled
3,Customer_3,Bangalore,Karnataka,India,2023-10-17,false,2023,10,8728,2024-07-19,939.67456,Cancelled
4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,false,2023,3,17286,2024-03-23,19.279629,Cancelled
4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,false,2023,3,2228,2024-11-23,512.2135,Delivered
6,Customer_6,Pune,Delhi,India,2023-08-29,false,2023,8,15577,2024-01-23,890.4308,Pending
6,Customer_6,Pune,Delhi,India,2023-08-29,false,2023,8,4985,2024-02-08,588.08203,Cancelled
7,Customer_7,Ahmedabad,West Bengal,India,2023-12-28,true,2023,12,139,2024-07-10,546.6385,Delivered
8,Customer_8,Pune,Karnataka,India,2023-06-22,true,2023,6,9347,2024-07-04,655.1634,Delivered
10,Customer_10,Pune,Gujarat,India,2023-08-05,true,2023,8,14646,2024-08-27,26.360586,Cancelled


In [0]:
# Total Orders Per Customer

customer_order_count = customers_order_df.groupBy('customer_id').count().orderBy(col('count').desc())
customer_order_count.display()

customer_id,count
11776,7
4294,6
5160,6
3884,6
13034,6
3336,6
7566,6
3243,6
14838,6
8612,5


In [0]:
customer_order_spent = customers_order_df.groupBy('customer_id').agg(sum('total_amount')).orderBy(col('sum(total_amount)').desc())


In [0]:
customer_order_spent.show(5)

+------------+-----+
|total_amount|count|
+------------+-----+
|     845.664|    2|
|    893.4693|    1|
|   132.83076|    1|
|    979.1805|    1|
|    96.99793|    1|
+------------+-----+
only showing top 5 rows



In [0]:
customer__avg_order_spent = customers_order_df.groupBy('customer_id').agg(avg('total_amount')).orderBy(col('avg(total_amount)').desc())


In [0]:
customer__avg_order_spent.show(10)

+-----------+-----------------+
|customer_id|avg(total_amount)|
+-----------+-----------------+
|      11854|   999.8642578125|
|         46| 999.592529296875|
|      17590|999.5726318359375|
|      11587|999.5595092773438|
|       6816|999.4348754882812|
|       9648|999.4214477539062|
|      15486| 999.034912109375|
|      10980|998.9071044921875|
|       1711| 998.773681640625|
|       6333|998.6394653320312|
+-----------+-----------------+
only showing top 10 rows



In [0]:
customer_order_count,customer_order_spent

Out[99]: (DataFrame[customer_id: int, count: bigint],
 DataFrame[customer_id: int, sum(total_amount): double])

In [0]:
# Customers with High Order Frequency but low total spend

customer_spend_vs_orders = customer_order_count.join(customer_order_spent,'customer_id','inner')\
    .orderBy(col('count').desc(),col('sum(total_amount)').desc())

In [0]:
customer_spend_vs_orders.show(55)

+-----------+-----+------------------+
|customer_id|count| sum(total_amount)|
+-----------+-----+------------------+
|      11776|    7|3438.3669471740723|
|       3336|    6| 4362.550720214844|
|       3884|    6|4187.9976806640625|
|       7566|    6|3647.1190643310547|
|      13034|    6|3195.0224742889404|
|      14838|    6| 2894.355628967285|
|       3243|    6| 2860.182746887207|
|       4294|    6| 1821.603916168213|
|       5160|    6|1656.7373294830322|
|      16020|    5| 3967.269256591797|
|      14372|    5|3961.7871704101562|
|      14933|    5|3828.5841674804688|
|      10559|    5| 3548.837860107422|
|       5425|    5|3389.1629333496094|
|      15014|    5|3365.0787658691406|
|      13940|    5| 3354.710479736328|
|       9282|    5| 3335.140525817871|
|       9073|    5| 3327.719924926758|
|       9415|    5|3314.7026977539062|
|       1636|    5|3195.5514945983887|
|       2330|    5| 3172.698211669922|
|      11537|    5|3155.4999084472656|
|       7762|    5|3116.1